In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import xarray as xr
from scipy.spatial.distance import euclidean, cdist

import sys

sys.path.append("..")

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from skimage.filters import gaussian, median

INFO:Constants:Logging to file: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/logs/Constants_20251119_135927.log
/tmp/ipykernel_773503/4157130849.py:33: DeprecationWarning: PlottingFunctions.Plotter is deprecated and will be removed in a future version. Use PlottingBase.PublicationPlotter directly instead.
  plotter = PlottingFunctions.Plotter()


In [2]:
# data_folder = '/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/old/'
data_folder = "../Camera_Calibrations/Ximea_Camera/"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])
image_size = 12
camera_parameters = {}
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
camera_parameters["gain"] = np.full((image_size, image_size), np.median(gain))
camera_parameters["offset"] = np.full((image_size, image_size), np.median(offset))
camera_parameters["variance"] = np.full((image_size, image_size), np.median(variance))
camera_parameters["readnoise"] = np.full((image_size, image_size), np.median(readnoise))
camera_parameters["rqe"] = np.full((image_size, image_size), np.median(rqe))
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ["B", "G", "R"]
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks

In [3]:
import types

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [4]:
notch_filter = "semrock-nf03-405-488-561-635e"
dichroic_mirror = "semrock-di03-r405-488-561-635-t1-25x36"
nilered_filter = "semrock-ff01-650-200-25"
shortpass_filter = "semrock-bsp01-785r"
dichroic_nored_mirror = "semrock-di03-r488-561-t1-25x36"
lp_561 = "semrock-blp02-561r"
#bp_584 = "semrock-ff01-582-64"

filters = [dichroic_nored_mirror, lp_561]
filter_spectra = S_F.get_dye_or_filter_data(
    names=filters, wavelength=wavelength, dye_or_filter=False
)

In [5]:
filter_spectra

array([[  2.74999999e-03,   3.56999994e-03,   1.35300001e-02, ...,
          9.70229983e-01,   9.70089972e-01,   9.71920013e-01],
       [  1.46130191e-07,   1.33928495e-07,   9.73773524e-08, ...,
          0.00000000e+00,   0.00000000e+00,   0.00000000e+00]])

In [6]:
S_F.get_pixel_fractions_dye_and_filters(filters=filters, dyes=['ATTO 565', 'ATTO 594'], pixel_QYs=pixel_QYs, wavelength=wavelength)

(array([ 618.05316526,  645.95819284]),
 array([[ 0.02833065,  0.33048532,  0.64118402],
        [ 0.03880403,  0.23301133,  0.72818465]]))

In [7]:
sigma = 1e-9
varepsilon = (sigma*6.023e23)/(np.log(10)*1e3)

In [8]:
varepsilon/1e11

2.6157556645032858

In [9]:
single_molecule_dyes = np.array(
    [
        ["ATTO 488", 2073],
        ["Alexa Fluor 488", 2811],
        ["CF488A", 3879],
        #["ATTO 594", 2000],
        ["Cy3B", 23195],
        ["ATTO 565", 11600],
        ["Janelia Fluor JF585-HaloTag conjugate", 2429],
        ["CF568", 13388],
        ["Cy5", 7801],
        ["Cy5B", 17090],
        ["ATTO 643", 23327],
        ["ATTO 647N", 18448],
        ["ATTO 655", 8273],
        ["CF640R", 12024],
        ["CF660R", 10399],
        ["abberior STAR 635", 12731],
        ["Janelia Fluor JF646-HaloTag conjugate", 14440],
        ["Alexa Fluor 647", 10348],
        ["Cy2", 6241],
        ["Cy3", 11022],
        ["Tetramethylrhodamine (TAMRA, TRITC)", 4884],
        ["Cy3.5", 4968],
        ["ATTO 647", 1526],
        ["ATTO 680", 1656],
        ["Cy5.5", 6337],
        ["Cy7", 852],
        ["Alexa Fluor 750", 703],
        ["ATTO 740", 779],
        ["Alexa Fluor 790", 740],
    ],
    dtype="object",
)

In [10]:
potential_dyes = [
    "Cy3B",
    #"ATTO 594",
    "CF550R",
    "Cy5B",
    "Alexa Fluor 488",
    "ATTO 488",
    "ATTO 643",
    "ATTO 647N",
    "ATTO 655",
    "Alexa Fluor 647",
    "CF640R",
    "CF660R",
    "abberior STAR 635",
    "ATTO 620",
    "ATTO 565",
    "CF568",
    "Janelia Fluor JF646-HaloTag conjugate",
    "ATTO 680",
    
]

In [11]:
import numpy as np
from src import Multicolour_Simulation_Functions
from src import SpectralFunctions

# Select optimal 5 dyes
result = MSF.optimal_dye_selector_simulated(
    potential_dyes=potential_dyes,
    single_molecule_dyes=single_molecule_dyes,
    filters=filters,
    smoothing_function=smoothing_function,
    camera_parameters=camera_parameters,
    wavelength=wavelength,
    n_dyes_desired=5,
    min_photons_per_100ms=500,
    n_simulations=10000,
    exhaustive_search=True,  # Use greedy for speed
    background_photons=50,
    verbose=True
)

print(f"\nOptimal dyes: {result['selected_dyes']}")
print(f"Classification accuracy: {result['overall_accuracy']:.1%}")


OPTIMAL DYE SELECTION VIA SIMULATION

Step 1: Filtering dyes (min 500 photons/100ms)...
  17 candidates -> 14 viable dyes
  Rejected: {'ATTO 620', 'CF550R', 'ATTO 680'}

Step 2-3: Simulating 10000 molecules per dye...
  Simulating Cy3B (23195 source / 4897 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.638, 0.333)
    Std  (A_R, A_G): (0.008, 0.008)
  Simulating Cy5B (17090 source / 3608 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.713, 0.220)
    Std  (A_R, A_G): (0.009, 0.008)
  Simulating Alexa Fluor 488 (2811 source / 593 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.576, 0.399)
    Std  (A_R, A_G): (0.026, 0.024)
  Simulating ATTO 488 (2073 source / 438 detector photons)...
    Fit success: 10000/10000 (100.0%)        

# Plot 1: Exemplar ternary plots from each selected dye

In [ ]:
# Plot 1: Exemplar ternary plots from each selected dye
from PlottingBase import PublicationPlotter

pub_plotter = PublicationPlotter()
n_dyes = len(result["selected_dyes"])

fig = plt.figure(figsize=(5*n_dyes, 4.5))

for i, dye_name in enumerate(result["selected_dyes"]):
    # Get simulation data for this dye
    sim_data = result["dye_simulations"][dye_name]
    A_R = sim_data["A_R"]
    A_G = sim_data["A_G"]
    A_B = sim_data["A_B"]
    
    # Remove NaN values
    valid = ~(np.isnan(A_R) | np.isnan(A_G) | np.isnan(A_B))
    A_R = A_R[valid]
    A_G = A_G[valid]
    A_B = A_B[valid]
    
    # Plot first 1000 examples on ternary plot
    n_plot = min(1000, len(A_R))
    
    ax = fig.add_subplot(1, n_dyes, i+1, projection="ternary")
    
    # Scatter plot
    ax.scatter(A_R[:n_plot], A_G[:n_plot], A_B[:n_plot], 
              s=3, alpha=0.4, c="steelblue", rasterized=True)
    
    # Calculate zoom limits based on distribution
    # For ternary plots: t (top) = R, l (left) = G, r (right) = B
    mean_R, std_R = A_R.mean(), A_R.std()
    mean_G, std_G = A_G.mean(), A_G.std()
    mean_B, std_B = A_B.mean(), A_B.std()
    
    # Use 5 sigma range for each component
    margin_factor = 5.0
    R_min = max(0.0, mean_R - margin_factor * std_R)
    R_max = min(1.0, mean_R + margin_factor * std_R)
    G_min = max(0.0, mean_G - margin_factor * std_G)
    G_max = min(1.0, mean_G + margin_factor * std_G)
    B_min = max(0.0, mean_B - margin_factor * std_B)
    B_max = min(1.0, mean_B + margin_factor * std_B)
    
    # Apply zoom using set_ternary_lim with proper constraint:
    # tmax + lmin + rmin = tmin + lmax + rmin = tmin + lmin + rmax = 1.0
    # This ensures the zoomed region forms a proper triangle
    ax.set_ternary_lim(
        R_min, R_max,  # tmin, tmax (Red/top)
        G_min, G_max,  # lmin, lmax (Green/left)
        B_min, B_max   # rmin, rmax (Blue/right)
    )
    
    # Set labels with colors
    ax.set_tlabel("Red", color="darkred", fontsize=7)
    ax.set_llabel("Green", color="darkgreen", fontsize=7)
    ax.set_rlabel("Blue", color="darkblue", fontsize=7)
    
    # Color axis lines
    ax.spines["lside"].set_color("darkred")
    ax.spines["rside"].set_color("darkgreen")
    ax.spines["tside"].set_color("darkblue")
    ax.spines["lside"].set_linewidth(1.5)
    ax.spines["rside"].set_linewidth(1.5)
    ax.spines["tside"].set_linewidth(1.5)
    
    # Tick parameters
    ax.taxis.set_tick_params(colors="darkred", labelsize=6)
    ax.laxis.set_tick_params(colors="darkgreen", labelsize=6)
    ax.raxis.set_tick_params(colors="darkblue", labelsize=6)
    
    # Grid
    ax.grid(True, alpha=0.3)
    
    # Title
    ax.set_title(f"{dye_name}\n({result["expected_photons"][dye_name]:.0f} photons)",
                fontsize=8, fontweight="bold", pad=15)

plt.tight_layout()
plt.show()


# Plot 2: Ternary density plot of ALL localizations from all viable dyes

In [ ]:
# Plot 2: Ternary density plot of ALL localizations from all viable dyes
from PlottingBase import PublicationPlotter

pub_plotter = PublicationPlotter()

# Collect all simulation data
all_A_R = []
all_A_G = []
all_A_B = []

for dye_name in result["dye_simulations"].keys():
    sim_data = result["dye_simulations"][dye_name]
    # Remove NaN values
    valid = ~(np.isnan(sim_data["A_R"]) | np.isnan(sim_data["A_G"]) | np.isnan(sim_data["A_B"]))
    all_A_R.extend(sim_data["A_R"][valid])
    all_A_G.extend(sim_data["A_G"][valid])
    all_A_B.extend(sim_data["A_B"][valid])

all_A_R = np.array(all_A_R)
all_A_G = np.array(all_A_G)
all_A_B = np.array(all_A_B)

print(f"Total localizations to plot: {len(all_A_R):,}")

# Use ternary density plot (hexbin)
fig, ax = pub_plotter.create_ternary_density(
    all_A_R, all_A_G, all_A_B,
    gridsize=100,
    cmap="Blues",
    title="All Simulated Dye Color Distributions",
    figsize=(8, 7),
    log_scale=True,
    show_colorbar=True
)

# Adjust font sizes
ax.set_tlabel("Red", color="darkred", fontsize=7)
ax.set_llabel("Green", color="darkgreen", fontsize=7)
ax.set_rlabel("Blue", color="darkblue", fontsize=7)
ax.taxis.set_tick_params(labelsize=6)
ax.laxis.set_tick_params(labelsize=6)
ax.raxis.set_tick_params(labelsize=6)
ax.set_title("All Simulated Dye Color Distributions", fontsize=8, fontweight="bold", pad=15)

plt.tight_layout()
plt.show()


# Plot 3: Top combinations displayed on ternary plots

In [ ]:
# Plot 3: Top combinations displayed on ternary plots
if result["all_combinations_tested"] is not None:
    # Exhaustive search mode - show top 10 combinations
    combos = result["all_combinations_tested"][:10]
    
    fig = plt.figure(figsize=(20, 8))
    
    for i, combo_result in enumerate(combos):
        combo_dyes = combo_result["dyes"]
        accuracy = combo_result["accuracy"]
        
        ax = fig.add_subplot(2, 5, i+1, projection="ternary")
        
        # Plot all dyes faintly in background
        for dye_name in result["dye_simulations"].keys():
            sim_data = result["dye_simulations"][dye_name]
            valid = ~(np.isnan(sim_data["A_R"]) | np.isnan(sim_data["A_G"]) | np.isnan(sim_data["A_B"]))
            n_plot = min(200, valid.sum())
            indices = np.random.choice(np.where(valid)[0], n_plot, replace=False)
            ax.scatter(sim_data["A_R"][indices], sim_data["A_G"][indices], sim_data["A_B"][indices],
                      s=1, alpha=0.05, c="gray", rasterized=True)
        
        # Highlight selected combination
        colors = plt.cm.tab10(range(len(combo_dyes)))
        for j, dye_name in enumerate(combo_dyes):
            if dye_name in result["dye_simulations"]:
                sim_data = result["dye_simulations"][dye_name]
                valid = ~(np.isnan(sim_data["A_R"]) | np.isnan(sim_data["A_G"]) | np.isnan(sim_data["A_B"]))
                n_plot = min(100, valid.sum())
                indices = np.random.choice(np.where(valid)[0], n_plot, replace=False)
                ax.scatter(sim_data["A_R"][indices], sim_data["A_G"][indices], sim_data["A_B"][indices],
                          s=8, alpha=0.7, c=[colors[j]], rasterized=True)
        
        # Styling
        ax.set_tlabel("", fontsize=1)  # Hide labels for compact layout
        ax.set_llabel("", fontsize=1)
        ax.set_rlabel("", fontsize=1)
        ax.taxis.set_tick_params(labelsize=5)
        ax.laxis.set_tick_params(labelsize=5)
        ax.raxis.set_tick_params(labelsize=5)
        
        # Color spines
        ax.spines["lside"].set_color("darkred")
        ax.spines["rside"].set_color("darkgreen")
        ax.spines["tside"].set_color("darkblue")
        
        ax.grid(True, alpha=0.2)
        
        rank_str = "BEST" if i == 0 else f"#{i+1}"
        ax.set_title(f"{rank_str}: {accuracy:.1%}",
                    fontsize=7, fontweight="bold" if i == 0 else "normal", pad=10)
    
    plt.suptitle("Top 10 Dye Combinations (Exhaustive Search)", 
                fontsize=10, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.show()
else:
    print("Greedy search mode - no combination history available")
    print("Rerun with exhaustive_search=True to visualize all tested combinations")


# Plot 4: Optimal dye selection displayed on ternary plot

In [ ]:
# Plot 4: Optimal dye selection displayed on ternary plot
from PlottingBase import PublicationPlotter

pub_plotter = PublicationPlotter()
fig = plt.figure(figsize=(10, 9))
ax = fig.add_subplot(111, projection="ternary")

selected_dyes = result["selected_dyes"]
colors = plt.cm.tab10(range(len(selected_dyes)))

# Plot each selected dye
for i, dye_name in enumerate(selected_dyes):
    sim_data = result["dye_simulations"][dye_name]
    
    # Remove NaN values
    valid = ~(np.isnan(sim_data["A_R"]) | np.isnan(sim_data["A_G"]) | np.isnan(sim_data["A_B"]))
    A_R = sim_data["A_R"][valid]
    A_G = sim_data["A_G"][valid]
    A_B = sim_data["A_B"][valid]
    
    # Plot simulation points
    n_plot = min(1000, len(A_R))
    ax.scatter(A_R[:n_plot], A_G[:n_plot], A_B[:n_plot],
              s=10, alpha=0.5, c=[colors[i]], label=dye_name, rasterized=True)
    
    # Annotate with dye number at mean position
    mean_R, mean_G, mean_B = A_R.mean(), A_G.mean(), A_B.mean()
    ax.text(mean_R, mean_G, mean_B, str(i+1), fontsize=10, fontweight="bold",
           ha="center", va="center", color="white",
           bbox=dict(boxstyle="circle", facecolor=colors[i], edgecolor="white", linewidth=2))

# Set labels with colors
ax.set_tlabel("Red", color="darkred", fontsize=7)
ax.set_llabel("Green", color="darkgreen", fontsize=7)
ax.set_rlabel("Blue", color="darkblue", fontsize=7)

# Color axis lines
ax.spines["lside"].set_color("darkred")
ax.spines["rside"].set_color("darkgreen")
ax.spines["tside"].set_color("darkblue")
ax.spines["lside"].set_linewidth(1.5)
ax.spines["rside"].set_linewidth(1.5)
ax.spines["tside"].set_linewidth(1.5)

# Tick parameters
ax.taxis.set_tick_params(colors="darkred", labelsize=6)
ax.laxis.set_tick_params(colors="darkgreen", labelsize=6)
ax.raxis.set_tick_params(colors="darkblue", labelsize=6)

# Grid
ax.grid(True, alpha=0.3)

# Title and legend
ax.set_title(f"Optimal {len(selected_dyes)}-Dye Combination\n"
            f"Overall Accuracy: {result["overall_accuracy"]:.1%}",
            fontsize=8, fontweight="bold", pad=15)
ax.legend(fontsize=6, loc=(1.05, 0.5), framealpha=0.9)

plt.tight_layout()
plt.show()


# Plot 5: Classification accuracy determination

In [ ]:
# Plot 5: Classification accuracy determination
from PlottingBase import PublicationPlotter

pub_plotter = PublicationPlotter()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Confusion Matrix
ax = axes[0]
confusion = result["confusion_matrix"]
selected_dyes = result["selected_dyes"]

im = ax.imshow(confusion, cmap="Blues", vmin=0, vmax=1, aspect="auto")

# Add text annotations
for i in range(len(selected_dyes)):
    for j in range(len(selected_dyes)):
        value = confusion[i, j]
        color = "white" if value > 0.5 else "black"
        ax.text(j, i, f"{value:.3f}", ha="center", va="center",
               fontsize=10, fontweight="bold", color=color)

ax.set_xticks(range(len(selected_dyes)))
ax.set_yticks(range(len(selected_dyes)))
ax.set_xticklabels([f"{i+1}" for i in range(len(selected_dyes))], fontsize=11)
ax.set_yticklabels([f"{i+1}" for i in range(len(selected_dyes))], fontsize=11)
ax.set_xlabel("Predicted Dye", fontsize=13, fontweight="bold")
ax.set_ylabel("True Dye", fontsize=13, fontweight="bold")
ax.set_title("Confusion Matrix
(Monte Carlo Classification)", 
            fontsize=14, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Classification Probability", fontsize=11)

# Panel B: Per-Dye Accuracy
ax = axes[1]
per_dye_accuracy = np.diag(confusion)
dye_labels = [f"{i+1}. {dye[:20]}" for i, dye in enumerate(selected_dyes)]

bars = ax.barh(range(len(selected_dyes)), per_dye_accuracy, 
              color=plt.cm.tab10(range(len(selected_dyes))), alpha=0.7)
ax.set_yticks(range(len(selected_dyes)))
ax.set_yticklabels(dye_labels, fontsize=10)
ax.set_xlabel("Classification Accuracy", fontsize=13, fontweight="bold")
ax.set_title("Per-Dye Classification Accuracy", fontsize=14, fontweight="bold")
ax.set_xlim(0.9, 1.0)
ax.grid(True, alpha=0.3, axis="x")

# Add value labels
for i, (bar, acc) in enumerate(zip(bars, per_dye_accuracy)):
    ax.text(acc - 0.002, bar.get_y() + bar.get_height()/2, 
           f"{acc:.1%}", ha="right", va="center", 
           fontsize=11, fontweight="bold", color="white")

plt.tight_layout()
plt.show()

# Print summary statistics
print("
" + "="*60)
print("CLASSIFICATION ACCURACY SUMMARY")
print("="*60)
for i, dye in enumerate(selected_dyes):
    acc = per_dye_accuracy[i]
    print(f"{i+1}. {dye:40s}: {acc:.2%}")
print(f"
Overall accuracy: {result["overall_accuracy"]:.2%}")
print("="*60)
